<a href="https://colab.research.google.com/github/ryankrismer/EM_field_solver/blob/main/EM_field_solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Electromagnetic field solver

In [13]:
import numpy as np
import tqdm
import sys

# Set threshold to infinity or system maximum
np.set_printoptions(threshold=sys.maxsize)

## Finding potentials from sources

In [28]:
# Defining constants and parameters

# E&M constants
c = 299792458                         # Speed of light in vacuum (m * s^-1)
epsilon_0 = 8.8541878188 * 10 ** -12  # Vacuum electric permissivity (F * m^-1)
mu_0 = 1.25663706127 * 10 ** -6       # Vacuum magnetic permeability (N * A^-2)

# General lattice parameters
Delta_t = 1.0                 # Difference between consecutive time coordinate values (s)
N_plot = 40 ** 4              # Number of observation points to be plotted
N_t = int(N_plot ** (1 / 4))  # Size of time coordinate axis

# Time lattice points
T = np.linspace(0.0, Delta_t * (N_t - 1), N_t)  # Time axis (s)

# Source lattice parameters
Delta_L_src = 1.0 # Distance between adjacent source spatial coordinate values (m)
n_src = 101       # Size of each source spatial coordinate axis (should be odd to ensure lattice centering)

# Observation lattice parameters
n_rad = 10  # Number of source radii from source sphere's surface to observe out to
N_obs = 2 * int(np.round(((3 * N_plot ** (3 / 4) / (4 * np.pi)) / (1 - 1 / (1 + n_rad) ** 3)) ** (1 / 3))) + 1  # Size of each observation spatial coordinate axis
# Note: N_obs should be odd so center of lattice is origin

In [ ]:
def potential_prereqs(src):
  """
  Function to perform prerequisite tasks for potential functions

  Parameter:
    src: Source distribution (4D numpy array) (axes are ordered as [t, x, y, z])

  Returns:
    n_i_src[1:4]: Sizes of source spatial coordinate axes (1D numpy array)
    r_src: Radius of spherical source (m) (float)
    R_obs: Radius of observation sphere (m) (float)
    P_src: Positional coordinate values for each source spatial axis (m) (1D numpy array)
    P_obs: Positional coordinate values for each observation spatial axis (m) (1D numpy array)
    Delta_L_obs: Distance between adjacent observation spatial coordinate values (m) (float)
    L_err: Uncertainty due to discreteness of spatial lattice (m) (float)
    src_sum: Placeholder array for source sums (4D numpy array)

  Note: Each 1D numpy array involved has format [x, y, z]
  """
  n_i_src = []

  for i in range(4):
    n_i_src.append(np.size(src, axis = i))  # Sizes of source coordinate axes

  # Ensuring size of time axis is same as expected
  if n_i_src[0] != N_t:
    raise ValueError(f"Source time axis does not match expected value of {N_t}")

  # Unpacking source axis sizes
  n_x_src = n_i_src[1]
  n_y_src = n_i_src[2]
  n_z_src = n_i_src[3]

  # Ensuring source is sphere
  if n_x_src != n_y_src or n_x_src != n_z_src or n_y_src != n_z_src:
    raise ValueError("Source spatial axis sizes are different. They must be equal for spherical source.")

  n_src = n_x_src

  # Ensuring n_src and N_obs are odd so lattice center is origin
  if n_src % 2 != 1:
    raise ValueError("n_src is not odd, but it must be odd for lattice centering")

  if N_obs % 2 != 1:
    raise ValueError("N_obs is not odd, but it must be odd for lattice centering")

  # Setting source lattice parameters
  r_src = (n_src - 1) / 2 * Delta_L_src  # Radius of spherical source (m)

  # Setting observation lattice parameters
  R_obs = r_src * (1 + n_rad) # Radius of observation sphere (m)

  # Spatial lattice points
  P_src = []
  P_obs = []

  for i in range(3):
    P_src.append(np.linspace(-r_src, r_src, n_src)) # Source spatial axes (m)
    P_obs.append(np.linspace(-R_obs, R_obs, N_obs)) # Observation spatial axes (m)

  # Setting general parameters
  Delta_L_obs = P_obs[0, 1] - P_obs[0, 0]               # Distance between adjacent observation spatial coordinate values (m)
  L_err = np.max([Delta_L_src, Delta_L_obs])            # Uncertainty due to discreteness of spatial lattice (m)

  src_sum = np.full((N_t, N_obs, N_obs, N_obs), np.nan) # Creating 4D array w/ set number of values per observation coordinate

  return n_i_src[1:4], r_src, R_obs, P_src, P_obs, Delta_L_obs, L_err, src_sum

In [3]:
def find_dist(r, r_ref):
  """
  Function to calculate the spatial distance between 2 coordinates

  Parameters:
    r: [x, y, z] (1D numpy array)
    r_ref: [x_ref, y_ref, z_ref] (1D numpy array)

  Returns: Distance between coordinates (float)
  """
  return np.sqrt((r[0] - r_ref[0]) ** 2 + (r[1] - r_ref[1]) ** 2 + (r[2] - r_ref[2]) ** 2)

In [ ]:
def causally_related(dist_src, t_i_src, t_i_obs, L_err):
  """
  Function to evaluate whether source point is causally related to observation point

  Parameters:
    dist_src: Distance between source and observation points (m) (float)
    t_i_src: Value of time for source point (s) (float)
    t_i_obs: Value of time for observation point (s) (float)
    L_err: Uncertainty due to discreteness of spatial lattice (m) (float)

  Returns: True if source point is related to observation point, False otherwise (boolean)
  """
  t_causal = dist_src / c + t_i_src

  return t_i_obs <= t_causal and t_i_obs > t_causal - L_err / c

In [29]:
def find_phi(Q):
  """
  Function to find scalar potential from charge distribution

  Parameter:
    Q: charge distribution (collection of point charges) (4D numpy array)

  Returns: scalar potential (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Performing prerequisites
  n_i_src, r_src, R_obs, P_src, P_obs, Delta_L_obs, L_err, lambda_sum = potential_prereqs(Q)

  # Unpacking source lattice parameters
  n_x_src = n_i_src[0]
  n_y_src = n_i_src[1]
  n_z_src = n_i_src[2]

  # Unpacking spatial lattice points
  X_src = P_src[0]  # Source x axis (m)
  Y_src = P_src[1]  # Source y axis (m)
  Z_src = P_src[2]  # Source z axis (m)

  X_obs = P_obs[0]  # Observation x axis (m)
  Y_obs = P_obs[1]  # Observation y axis (m)
  Z_obs = P_obs[2]  # Observation z axis (m)

  # Calculating and setting values for each plottable observation point
  for t_obs in tqdm.tqdm(range(N_t)):
    t_i_obs = T[t_obs]

    for x_obs in tqdm.tqdm(range(N_obs)):
      break_x = False
      x_i_obs = X_obs[x_obs]

      for y_obs in range(N_obs):
        y_i_obs = Y_obs[y_obs]
        dist_x_y_origin = find_dist([x_i_obs, y_i_obs, 0.0], [0.0, 0.0, 0.0]) # Distance of (x, y) from origin (m)

        if dist_x_y_origin > R_obs:                                           # True if (x, y) line is beyond plotting region
          if x_i_obs > 0.0:                                 # True if all future x values will also be beyond plotting region
            break_x = True
            break

          if y_i_obs > 0.0:                                 # True if all future y values will also be beyond plotting region
            break

          continue                                          # Skip this y value but continue iterating through y

        for z_obs in range(N_obs):
          valid_obs_point = False                           # Default to not summing up source contributions
          z_i_obs = Z_obs[z_obs]
          dist_origin = find_dist([x_i_obs, y_i_obs, z_i_obs], [0.0, 0.0, 0.0]) # Distance from origin (m)

          if dist_origin <= R_obs and dist_origin > r_src:  # Ensuring we only save values we're going to plot
            lambda_i = []

            # Iterating through all source points that occur at an earlier time than observation time
            for t_src in range(t_obs + 1):
              t_i_src = T[t_src]

              for x_src in range(n_x_src):
                x_i_src = X_src[x_src]

                for y_src in range(n_y_src):
                  y_i_src = Y_src[y_src]

                  for z_src in range(n_z_src):
                    if np.isnan(Q[t_src, x_src, y_src, z_src]): # Ensuring we don't check empty points
                      continue

                    z_i_src = Z_src[z_src]
                    dist_src = find_dist([x_i_obs, y_i_obs, z_i_obs], [x_i_src, y_i_src, z_i_src])  # Distance from src (m)

                    # Only contributions are source points causally related to observation points
                    if causally_related(dist_src, t_i_src, t_i_obs, L_err):
                      valid_obs_point = True
                      lambda_i.append(Q[t_src, x_src, y_src, z_src] / dist_src)

            if valid_obs_point:
              lambda_sum[t_obs, x_obs, y_obs, z_obs] = np.sum(lambda_i) # Adding up all the source contributions

      if break_x: # Break out of x loop if all future x values will be beyond plotting region
        break

  return Delta_t / (4 * np.pi * epsilon_0) * lambda_sum

In [ ]:
def find_A_i(I, i):
  """
  Function to find component of vector potential from current distribution

  Parameters:
    I: current distribution (collection of current segments) (4D numpy array)
    i: potential component desired

  Returns: i component of vector potential (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Performing prerequisites
  n_i_src, r_src, R_obs, P_src, P_obs, Delta_L_obs, L_err, current_sum = potential_prereqs(I)

  # Unpacking source lattice parameters
  n_x_src = n_i_src[0]
  n_y_src = n_i_src[1]
  n_z_src = n_i_src[2]

  # Unpacking spatial lattice points
  X_src = P_src[0]  # Source x axis (m)
  Y_src = P_src[1]  # Source y axis (m)
  Z_src = P_src[2]  # Source z axis (m)

  X_obs = P_obs[0]  # Observation x axis (m)
  Y_obs = P_obs[1]  # Observation y axis (m)
  Z_obs = P_obs[2]  # Observation z axis (m)

  # Calculating and setting values for each plottable observation point
  for t_obs in tqdm.tqdm(range(N_t)):
    t_i_obs = T[t_obs]

    for x_obs in tqdm.tqdm(range(N_obs)):
      break_x = False
      x_i_obs = X_obs[x_obs]

      for y_obs in range(N_obs):
        y_i_obs = Y_obs[y_obs]
        dist_x_y_origin = find_dist([x_i_obs, y_i_obs, 0.0], [0.0, 0.0, 0.0]) # Distance of (x, y) from origin (m)

        if dist_x_y_origin > R_obs:                                           # True if (x, y) line is beyond plotting region
          if x_i_obs > 0.0:                                 # True if all future x values will also be beyond plotting region
            break_x = True
            break

          if y_i_obs > 0.0:                                 # True if all future y values will also be beyond plotting region
            break

          continue                                          # Skip this y value but continue iterating through y

        for z_obs in range(N_obs):
          valid_obs_point = False                           # Default to not summing up source contributions
          z_i_obs = Z_obs[z_obs]
          dist_origin = find_dist([x_i_obs, y_i_obs, z_i_obs], [0.0, 0.0, 0.0]) # Distance from origin (m)

          if dist_origin <= R_obs and dist_origin > r_src:  # Ensuring we only save values we're going to plot
            current_i = []

            # Iterating through all source points that occur at an earlier time than observation time
            for t_src in range(t_obs + 1):
              t_i_src = T[t_src]

              for x_src in range(n_x_src):
                x_i_src = X_src[x_src]

                for y_src in range(n_y_src):
                  y_i_src = Y_src[y_src]

                  for z_src in range(n_z_src):
                    if np.isnan(I[t_src, x_src, y_src, z_src]): # Ensuring we don't check empty points
                      continue

                    z_i_src = Z_src[z_src]
                    dist_src = find_dist([x_i_obs, y_i_obs, z_i_obs], [x_i_src, y_i_src, z_i_src])  # Distance from src (m)

                    # Only contributions are source points causally related to observation points
                    if causally_related(dist_src, t_i_src, t_i_obs, L_err):
                      valid_obs_point = True
                      current_i.append(I[t_src, x_src, y_src, z_src] / dist_src)

            if valid_obs_point:
              current_sum[t_obs, x_obs, y_obs, z_obs] = np.sum(current_i) # Adding up all the source contributions

      if break_x: # Break out of x loop if all future x values will be beyond plotting region
        break

  return mu_0 * Delta_t * Delta_L_src / (4 * np.pi) * current_sum

### Test of scalar potential function

In [32]:
n_src = 9
Q = np.random.rand(N_t, n_src, n_src, n_src)

# Setting source lattice parameters
r_src = (n_src - 1) / 2 * Delta_L_src  # Radius of spherical source (m)

# Source spatial lattice points
X_src = np.linspace(-r_src, r_src, n_src) # Source x axis (m)
Y_src = X_src # Source y axis (m)
Z_src = X_src # Source z axis (m)

# Masking points outside sphere
for t in tqdm.tqdm(range(N_t)):
  for x in range(n_src):
    for y in range(n_src):
      for z in range(n_src):
        x_i = X_src[x]
        y_i = Y_src[y]
        z_i = Z_src[z]
        dist = find_dist([x_i, y_i, z_i], [0.0, 0.0, 0.0])

        if dist > r_src:
          Q[t, x, y, z] = np.nan

np.save(f"Q_test_{n_src}.npy", Q)

100%|██████████| 40/40 [00:00<00:00, 488.69it/s]


In [33]:
phi = find_phi(Q)
np.save(f"phi_test_{n_src}.npy", phi)

  0%|          | 0/40 [00:01<?, ?it/s]


KeyboardInterrupt: 

## Finding fields from potentials

In [ ]:
def find_E_i(phi, A, i):
  """
  Function to find E field component from potentials phi and A_i

  Parameters:
    phi: scalar potential (4D numpy array)
    A: [A_x, A_y, A_z] components of vector potential (each is 4D numpy array)
    i: field component desired

  Returns: i component of E field (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  if i != 1 and i != 2 and i != 3:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A components based on index values
  A_i = A[i - 1]

  return -np.gradient(phi, axis = i) - np.gradient(A_i, axis = 0)

In [ ]:
def find_B_i(A, i):
  """
  Function to find B field component from potential A

  Parameters:
    A: [A_x, A_y, A_z] components of vector potential (each is 4D numpy array)
    i: field component desired

  Returns: i component of B field (4D numpy array)

  Note: For all 4D arrays involved, axes are ordered as [t, x, y, z]
  """
  # Finding index values to ensure cyclicity
  if i == 1:
    j = 2
    k = 3
  elif i == 2:
    j = 3
    k = 1
  elif i == 3:
    j = 1
    k = 2
  else:
    raise ValueError("i value not allowed. i must be a spatial index within [1, 2, 3]")

  # Finding A components based on index values
  A_j = A[j - 1]
  A_k = A[k - 1]

  return np.gradient(A_k, axis = j) - np.gradient(A_j, axis = k)

In [ ]:
# Test of find_E_i() w/ test phi
A_x = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save(f"A_x_test_{n_src}.npy", A_x)
A_y = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save(f"A_y_test_{n_src}.npy", A_y)
A_z = np.random.rand(N_t, N_obs, N_obs, N_obs)
np.save(f"A_z_test_{n_src}.npy", A_z)
E_x = find_E_i(phi, [A_x, A_y, A_z], 1)
np.save(f"E_x_test_{n_src}.npy", E_x)

In [ ]:
# # Test of find_B_i()
# A_x = np.random.rand(3, 3, 3, 3)
# A_y = np.random.rand(3, 3, 3, 3)
# A_z = np.random.rand(3, 3, 3, 3)
# B_x = find_B_i([A_x, A_y, A_z], 1)
# print(f"A_x is {A_x}")
# print(f"A_y is {A_y}")
# print(f"A_z is {A_z}")
# print(f"B_x is {B_x}")

A_x is [[[[0.95307154 0.3869769  0.81464471]
   [0.92625786 0.48318043 0.09935442]
   [0.65977942 0.90902888 0.58763308]]

  [[0.78057705 0.02643401 0.23195566]
   [0.0043902  0.27951875 0.26583555]
   [0.57816752 0.5581665  0.43904707]]

  [[0.04150032 0.05817008 0.25300806]
   [0.72436458 0.44803979 0.56860463]
   [0.97810475 0.51435    0.91890006]]]


 [[[0.80400318 0.20275559 0.40467763]
   [0.17694077 0.95141178 0.59619766]
   [0.81901117 0.91792293 0.59824475]]

  [[0.61722773 0.00798019 0.67910435]
   [0.0899428  0.99873084 0.36475039]
   [0.55197416 0.25923369 0.59883985]]

  [[0.64473253 0.56582509 0.05307403]
   [0.85658534 0.97483502 0.62670432]
   [0.94130241 0.14511224 0.2623221 ]]]


 [[[0.11700951 0.27564612 0.96687832]
   [0.29951137 0.2632959  0.52804578]
   [0.60705034 0.98712846 0.89032515]]

  [[0.00273272 0.99889682 0.60025552]
   [0.42531668 0.14638614 0.30821971]
   [0.4559946  0.95861658 0.51063308]]

  [[0.12249758 0.98592893 0.11122947]
   [0.21745324 0.924399